# RetNet-300M: Inference from the Latest Checkpoint

Loads the newest valid `ckpt_step_*.pt` from the Drive folder (`MyDrive/retnet-300m/runs/<run>/`) and
generates text with the calculator in the loop. Works on a GPU or a CPU runtime (CPU is slower).

Needs the same Drive setup as the training notebook: `retnet-300m-colab.zip` and `runs/` in `MyDrive/retnet-300m/`.
Safe to run while training is in progress: it only reads checkpoints.

In [ ]:
# 1. Mount Drive, unpack code
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib, subprocess, sys, shutil

FOLDER_NAME = "retnet-300m"
MOUNT = pathlib.Path("/content/drive")
candidates = [MOUNT / "MyDrive" / FOLDER_NAME]
for root in ("Shareddrives", ".shortcut-targets-by-id"):
    if (MOUNT / root).is_dir():
        candidates += [d / FOLDER_NAME for d in sorted((MOUNT / root).iterdir()) if d.is_dir()]
DRIVE = next((c for c in candidates if c.is_dir()), MOUNT / "MyDrive" / FOLDER_NAME)
BUNDLE = DRIVE / "retnet-300m-colab.zip"
RUNS = DRIVE / "runs"
assert BUNDLE.is_file(), f"Could not find {BUNDLE}. See the setup steps in the training notebook."

CODE = pathlib.Path("/content/retnet-300m")
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["unzip", "-oq", str(BUNDLE), "-d", str(CODE)], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tokenizers"], check=True)
sys.path.insert(0, str(CODE))
print((CODE / "BUILD_INFO.txt").read_text().strip() if (CODE / "BUILD_INFO.txt").is_file() else "no BUILD_INFO.txt")
print("Runs on Drive:", sorted(p.name for p in RUNS.iterdir() if p.is_dir()) if RUNS.is_dir() else "none")

In [ ]:
# 2. Load the newest valid checkpoint (falls back to the previous one if the newest upload was cut off)
import torch
from retnet.checkpoint import checkpoint_candidates, load_run

RUN_NAME = "retnet-300m"  # any folder listed above
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
run_dir = RUNS / RUN_NAME
assert checkpoint_candidates(run_dir), f"No checkpoint in {run_dir} yet. Train first (training notebook, cell 6)."

model, tok, step = load_run(run_dir, DEVICE)
print(f"Loaded {RUN_NAME} step {step} ({model.num_params(False) / 1e6:.0f}M params) on "
      f"{torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}")

In [ ]:
# 3. Generation helper
import time
from retnet.calculator import generate_with_calculator

STOPS = ("\nQuestion:", "\nProblem:")

def ask(text, mode="math", max_new_tokens=320, temperature=0.0, top_k=50, calculator=True):
    """mode: "math" -> 'Question: ...\nAnswer:', "code" -> 'Problem: ...\n\nSolution (Python):', "raw" -> text as is.
    temperature=0 is greedy; try 0.7 for varied samples."""
    prompt = {"math": f"Question: {text}\nAnswer:",
              "code": f"Problem:\n{text}\n\nSolution (Python):\n",
              "raw": text}[mode]
    t0 = time.time()
    with torch.inference_mode():
        out, calls = generate_with_calculator(model, tok, prompt, max_new_tokens, temperature, top_k,
                                              stop=STOPS, use_calculator=calculator)
    for s in STOPS:
        out = out.removesuffix(s)
    print(prompt + out)
    print(f"\n[{calls} calculator call(s), {time.time() - t0:.1f}s]")
    return out

In [ ]:
# 4. Examples
ask("A baker makes 24 muffins per tray and bakes 7 trays. He sells 150 muffins. How many are left?")
print("\n" + "=" * 80 + "\n")
ask("Write a Python function `binary_search(arr, target)` that returns the index of `target` in sorted list `arr`, or -1 if not found.", mode="code")
print("\n" + "=" * 80 + "\n")
ask("Photosynthesis is the process by which", mode="raw", temperature=0.7)

In [ ]:
# 5. Interactive: type a question. Prefix "/code " for coding, ">" for raw text. Empty line to stop.
while True:
    line = input("> ").strip()
    if not line:
        break
    if line.startswith("/code "):
        ask(line[6:], mode="code")
    elif line.startswith(">"):
        ask(line[1:].lstrip(), mode="raw")
    else:
        ask(line)
    print()